In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)
from IPython.display import display, Markdown

X_train = pd.read_pickle('../data/processed/tree_ready/X_train.pkl')
X_test = pd.read_pickle('../data/processed/tree_ready/X_test.pkl')
y_train = pd.read_pickle('../data/processed/tree_ready/y_train.pkl')
y_test = pd.read_pickle('../data/processed/tree_ready/y_test.pkl')

results = []
interpretation_log = []  # collects text for saving

def evaluate_and_interpret(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    metrics = {'model': name, 'accuracy': acc, 'precision': prec,
               'recall': rec, 'f1': f1, 'auc': auc}
    results.append(metrics)

    # --- Auto-generated plain-language interpretation ---
    recall_quality = "strong" if rec > 0.7 else "moderate" if rec > 0.5 else "weak"
    auc_quality = "strong" if auc > 0.8 else "moderate" if auc > 0.65 else "weak"

    md = f"""### {name}

**Metrics:**
- Accuracy: {acc:.3f}
- Precision: {prec:.3f}
- Recall (Sensitivity): {rec:.3f}
- F1 Score: {f1:.3f}
- AUC: {auc:.3f}

**Confusion Matrix:** TN={tn}, FP={fp}, FN={fn}, TP={tp}

**Interpretation:**
- The model correctly identifies {rec*100:.1f}% of truly anemic women (recall) — this is {recall_quality} for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, {prec*100:.1f}% actually are (precision).
- AUC of {auc:.3f} indicates {auc_quality} ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): {fn} — {"a concerning number given clinical implications" if fn > tp*0.3 else "a relatively acceptable number, though always worth minimizing further"}.
"""
    display(Markdown(md))
    interpretation_log.append(md)
    return metrics

In [2]:
X_train_scaled = pd.read_pickle('../data/processed/neural_ready/X_train.pkl')
X_test_scaled = pd.read_pickle('../data/processed/neural_ready/X_test.pkl')

log_reg = LogisticRegression(class_weight='balanced', max_iter=5000, solver='saga', random_state=42)
log_reg.fit(X_train_scaled, y_train)
evaluate_and_interpret('Logistic Regression', log_reg, X_test_scaled, y_test)

### Logistic Regression

**Metrics:**
- Accuracy: 0.673
- Precision: 0.484
- Recall (Sensitivity): 0.573
- F1 Score: 0.525
- AUC: 0.693

**Confusion Matrix:** TN=332, FP=130, FN=91, TP=122

**Interpretation:**
- The model correctly identifies 57.3% of truly anemic women (recall) — this is moderate for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 48.4% actually are (precision).
- AUC of 0.693 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 91 — a concerning number given clinical implications.


{'model': 'Logistic Regression',
 'accuracy': 0.6725925925925926,
 'precision': 0.48412698412698413,
 'recall': 0.5727699530516432,
 'f1': 0.524731182795699,
 'auc': 0.6933875983171758}

In [3]:
dt = DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42)
dt.fit(X_train, y_train)
evaluate_and_interpret('Decision Tree', dt, X_test, y_test)

### Decision Tree

**Metrics:**
- Accuracy: 0.693
- Precision: 0.514
- Recall (Sensitivity): 0.531
- F1 Score: 0.522
- AUC: 0.679

**Confusion Matrix:** TN=355, FP=107, FN=100, TP=113

**Interpretation:**
- The model correctly identifies 53.1% of truly anemic women (recall) — this is moderate for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 51.4% actually are (precision).
- AUC of 0.679 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 100 — a concerning number given clinical implications.


{'model': 'Decision Tree',
 'accuracy': 0.6933333333333334,
 'precision': 0.5136363636363637,
 'recall': 0.5305164319248826,
 'f1': 0.5219399538106235,
 'auc': 0.6793081722659187}

In [4]:
knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train, y_train)
evaluate_and_interpret('KNN', knn, X_test, y_test)

### KNN

**Metrics:**
- Accuracy: 0.708
- Precision: 0.565
- Recall (Sensitivity): 0.329
- F1 Score: 0.415
- AUC: 0.700

**Confusion Matrix:** TN=408, FP=54, FN=143, TP=70

**Interpretation:**
- The model correctly identifies 32.9% of truly anemic women (recall) — this is weak for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 56.5% actually are (precision).
- AUC of 0.700 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 143 — a concerning number given clinical implications.


{'model': 'KNN',
 'accuracy': 0.7081481481481482,
 'precision': 0.5645161290322581,
 'recall': 0.3286384976525822,
 'f1': 0.41543026706231456,
 'auc': 0.6995660833689002}

In [7]:
# Save all interpretations as one .txt file
output_path = '../data/processed/baseline_model_interpretation.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('\n\n---\n\n'.join(interpretation_log))
print(f"Saved interpretations to {output_path}")

# Save the metrics table too, for the model comparison notebook later
results_df = pd.DataFrame(results)
results_df.to_csv('../results/metrics/baseline_metrics.csv', index=False)
print(results_df)

Saved interpretations to ../data/processed/baseline_model_interpretation.txt
                 model  accuracy  precision    recall        f1       auc
0  Logistic Regression  0.672593   0.484127  0.572770  0.524731  0.693388
1        Decision Tree  0.693333   0.513636  0.530516  0.521940  0.679308
2                  KNN  0.708148   0.564516  0.328638  0.415430  0.699566


In [8]:
knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train_scaled, y_train)
evaluate_and_interpret('KNN (scaled)', knn, X_test_scaled, y_test)

### KNN (scaled)

**Metrics:**
- Accuracy: 0.689
- Precision: 0.513
- Recall (Sensitivity): 0.286
- F1 Score: 0.367
- AUC: 0.659

**Confusion Matrix:** TN=404, FP=58, FN=152, TP=61

**Interpretation:**
- The model correctly identifies 28.6% of truly anemic women (recall) — this is weak for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 51.3% actually are (precision).
- AUC of 0.659 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 152 — a concerning number given clinical implications.


{'model': 'KNN (scaled)',
 'accuracy': 0.6888888888888889,
 'precision': 0.5126050420168067,
 'recall': 0.2863849765258216,
 'f1': 0.3674698795180723,
 'auc': 0.6590146942259617}

In [9]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# SVM — scale-sensitive (like KNN/LR), use neural_ready data
svm = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
evaluate_and_interpret('SVM', svm, X_test_scaled, y_test)

# Regularized Logistic Regression (L1/Lasso) — also scale-sensitive
log_reg_l1 = LogisticRegression(penalty='l1', solver='liblinear', class_weight='balanced',
                                  C=0.5, random_state=42)
log_reg_l1.fit(X_train_scaled, y_train)
evaluate_and_interpret('Logistic Regression (L1 Regularized)', log_reg_l1, X_test_scaled, y_test)

### SVM

**Metrics:**
- Accuracy: 0.677
- Precision: 0.490
- Recall (Sensitivity): 0.577
- F1 Score: 0.530
- AUC: 0.688

**Confusion Matrix:** TN=334, FP=128, FN=90, TP=123

**Interpretation:**
- The model correctly identifies 57.7% of truly anemic women (recall) — this is moderate for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 49.0% actually are (precision).
- AUC of 0.688 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 90 — a concerning number given clinical implications.


### Logistic Regression (L1 Regularized)

**Metrics:**
- Accuracy: 0.671
- Precision: 0.482
- Recall (Sensitivity): 0.573
- F1 Score: 0.524
- AUC: 0.688

**Confusion Matrix:** TN=331, FP=131, FN=91, TP=122

**Interpretation:**
- The model correctly identifies 57.3% of truly anemic women (recall) — this is moderate for a health screening context, where missing an anemic case (false negative) is costly.
- Of women predicted anemic, 48.2% actually are (precision).
- AUC of 0.688 indicates moderate ability to distinguish anemic from non-anemic women overall.
- False negatives (missed anemia cases): 91 — a concerning number given clinical implications.


{'model': 'Logistic Regression (L1 Regularized)',
 'accuracy': 0.6711111111111111,
 'precision': 0.48221343873517786,
 'recall': 0.5727699530516432,
 'f1': 0.5236051502145923,
 'auc': 0.6878798040769871}

In [10]:
# Save all interpretations
output_path = '../data/processed/baseline_model_interpretation.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('\n\n---\n\n'.join(interpretation_log))
print(f"Saved interpretations to {output_path}")

results_df = pd.DataFrame(results)
results_df.to_csv('../results/metrics/baseline_metrics.csv', index=False)
print(results_df)

Saved interpretations to ../data/processed/baseline_model_interpretation.txt
                                  model  accuracy  precision    recall  \
0                   Logistic Regression  0.672593   0.484127  0.572770   
1                         Decision Tree  0.693333   0.513636  0.530516   
2                                   KNN  0.708148   0.564516  0.328638   
3                          KNN (scaled)  0.688889   0.512605  0.286385   
4                                   SVM  0.677037   0.490040  0.577465   
5  Logistic Regression (L1 Regularized)  0.671111   0.482213  0.572770   

         f1       auc  
0  0.524731  0.693388  
1  0.521940  0.679308  
2  0.415430  0.699566  
3  0.367470  0.659015  
4  0.530172  0.688083  
5  0.523605  0.687880  


In [12]:
# Save trained baseline models for later reuse
with open('../models/ml/logistic_regression.pkl', 'wb') as f:
    pickle.dump(log_reg, f)
with open('../models/ml/decision_tree.pkl', 'wb') as f:
    pickle.dump(dt, f)
with open('../models/ml/knn.pkl', 'wb') as f:
    pickle.dump(knn, f)
with open('../models/ml/svm.pkl', 'wb') as f:
    pickle.dump(svm, f)